In [29]:
import requests
from langchain_openai.chat_models import ChatOpenAI
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from pydantic import BaseModel, Field
from google.cloud.firestore import Client

In [4]:
response = requests.get("https://fantasy.premierleague.com/api/bootstrap-static/")
response.raise_for_status()
response_dict = response.json()
for event in response_dict.get("events"):
    if event["is_current"]:
        gw_id = event["id"]
        is_complete = event["finished"]

In [5]:
response = requests.get(f"https://fantasy.premierleague.com/api/leagues-classic/{161855}/standings/")
response.raise_for_status()
response_dict = response.json()
league_name = response_dict["league"]["name"]
teams = []
for team in response_dict["standings"]["results"]:
    player_dict = {}
    player_dict["Team Name"] = team.get("entry_name","")
    player_dict["Manager"] = team.get("player_name","")
    player_dict["Rank"] = team.get("rank","")
    player_dict["Previous Rank"] = team.get("last_rank","")
    player_dict["Points"] = team.get("total","")
    player_dict["Week Points"] = team.get("event_total","")
    player_dict["ID"] = team.get("entry",0)
    teams.append(player_dict)
teams

[{'Team Name': 'Major League Saka',
  'Manager': 'Sonny Spencer',
  'Rank': 1,
  'Previous Rank': 0,
  'Points': 85,
  'Week Points': 85,
  'ID': 878373},
 {'Team Name': 'Back of the Neto',
  'Manager': 'Julian Hoyle',
  'Rank': 2,
  'Previous Rank': 0,
  'Points': 73,
  'Week Points': 73,
  'ID': 2773133},
 {'Team Name': 'Øde-god',
  'Manager': 'Lucas Wood-Oliván',
  'Rank': 3,
  'Previous Rank': 0,
  'Points': 69,
  'Week Points': 69,
  'ID': 6056871},
 {'Team Name': 'Money Muniz',
  'Manager': 'Giles Usher',
  'Rank': 4,
  'Previous Rank': 0,
  'Points': 50,
  'Week Points': 50,
  'ID': 1070778},
 {'Team Name': '50 Shades of Gray',
  'Manager': 'Marc Tatam',
  'Rank': 5,
  'Previous Rank': 0,
  'Points': 49,
  'Week Points': 49,
  'ID': 29896}]

In [6]:
for team in teams:
    response = requests.get(f"https://fantasy.premierleague.com/api/entry/{team.get('ID')}/event/{gw_id}/picks/")
    response.raise_for_status()
    response_dict = response.json()
    team["Starting 11"] = response_dict["picks"][:11]
    team["Subs"] = response_dict["picks"][11:]
    team["Bench Points"] = response_dict["entry_history"]["points_on_bench"]

In [7]:
response = requests.get(f"https://fantasy.premierleague.com/api/bootstrap-static/")
response.raise_for_status()
response_dict = response.json()
players = response_dict["elements"]
for team in teams:
    for selected in team["Starting 11"]:
        for player in players:
            if player["id"] == selected.get("element"):
                name = player["first_name"] + " " + player["second_name"]
                selected["Name"] = name
                selected["Points"] = player["event_points"]
        if selected["is_captain"]:
            team["Captain"] = selected["Name"]
        if selected["is_vice_captain"]:
            team["Vice Captain"] = selected["Name"]
    for selected in team["Subs"]:
        for player in players:
            if player["id"] == selected.get("element"):
                name = player["first_name"] + " " + player["second_name"]
                selected["Name"] = name
                selected["Points"] = player["event_points"]
teams

[{'Team Name': 'Major League Saka',
  'Manager': 'Sonny Spencer',
  'Rank': 1,
  'Previous Rank': 0,
  'Points': 85,
  'Week Points': 85,
  'ID': 878373,
  'Starting 11': [{'element': 15,
    'position': 1,
    'multiplier': 1,
    'is_captain': False,
    'is_vice_captain': False,
    'Name': 'David Raya Martin',
    'Points': 8},
   {'element': 395,
    'position': 2,
    'multiplier': 1,
    'is_captain': False,
    'is_vice_captain': False,
    'Name': 'Dan Burn',
    'Points': 5},
   {'element': 18,
    'position': 3,
    'multiplier': 1,
    'is_captain': False,
    'is_vice_captain': False,
    'Name': 'William Saliba',
    'Points': 6},
   {'element': 311,
    'position': 4,
    'multiplier': 1,
    'is_captain': False,
    'is_vice_captain': True,
    'Name': 'Trent Alexander-Arnold',
    'Points': 8},
   {'element': 398,
    'position': 5,
    'multiplier': 1,
    'is_captain': False,
    'is_vice_captain': False,
    'Name': 'Anthony Gordon',
    'Points': 3},
   {'element':

In [8]:
team_template = """
<Team>
Team Name: {team_name}

League Rank: {team_rank}
Previous League Rank: {team_old_rank}

Points: {team_points}
This Weeks Points: {team_week_points}

Manager: {team_manager}

Starting 11:

{starting_11}

Bench: 

{bench}

Points on the bench: {bench_points}

Captain: {captain}
Vice Captain: {vice_captain}
</Team>
"""

player_template = "Player Name: {player_name}, Player Points:{player_points}"

In [9]:
formatted_teams = []
for team in teams:
    formatted_starters = []
    for player in team["Starting 11"]:
        formatted_starters.append(player_template.format(player_name=player["Name"], player_points=player["Points"]))
    formatted_bench = []
    for player in team["Subs"]:
        formatted_bench.append(player_template.format(player_name=player["Name"], player_points=player["Points"]))
    formatted_teams.append(team_template.format(team_name=team["Team Name"],
                                                team_rank=team["Rank"],
                                                team_old_rank=team["Previous Rank"],
                                                team_points=team["Points"],
                                                team_week_points=team["Week Points"],
                                                bench_points=team["Bench Points"],
                                                team_manager=team["Manager"],
                                                captain=team["Captain"],
                                                vice_captain=team["Vice Captain"],
                                                starting_11="\n".join(formatted_starters),
                                                bench="\n".join(formatted_bench)
                                                ))
formatted_teams
    

['\n<Team>\nTeam Name: Major League Saka\n\nLeague Rank: 1\nPrevious League Rank: 0\n\nPoints: 85\nThis Weeks Points: 85\n\nManager: Sonny Spencer\n\nStarting 11:\n\nPlayer Name: David Raya Martin, Player Points:8\nPlayer Name: Dan Burn, Player Points:5\nPlayer Name: William Saliba, Player Points:6\nPlayer Name: Trent Alexander-Arnold, Player Points:8\nPlayer Name: Anthony Gordon, Player Points:3\nPlayer Name: Amad Diallo, Player Points:3\nPlayer Name: Mohamed Salah, Player Points:14\nPlayer Name: Emile Smith Rowe, Player Points:3\nPlayer Name: Alexander Isak, Player Points:5\nPlayer Name: Erling Haaland, Player Points:7\nPlayer Name: Chris Wood, Player Points:9\n\nBench: \n\nPlayer Name: Łukasz Fabiański, Player Points:0\nPlayer Name: Valentín Barco, Player Points:0\nPlayer Name: Joachim Andersen, Player Points:0\nPlayer Name: Harry Winks, Player Points:2\n\nPoints on the bench: 2\n\nCaptain: Mohamed Salah\nVice Captain: Trent Alexander-Arnold\n</Team>\n',
 '\n<Team>\nTeam Name: Back 

In [21]:
full_prompt_template = """Generate a report in the style of a new outlet covering the weekends football events for my fantasy football league. Make the headline as eyecatching as possible.

The league name is {league_name} and we are in week {gameweek} of 38.

Here are the teams:

{teams}

Note that the captains points are doubled.
"""

In [13]:
full_prompt_template.format(league_name=league_name, gameweek=gw_id, teams="\n".join(formatted_teams))

'Generate a report in the style of a new outlet covering the weekends football events for my fantasy football league.\n\nThe league name is European Sigma League and we are in week 1 of 38.\n\nHere are the teams:\n\n\n<Team>\nTeam Name: Major League Saka\n\nLeague Rank: 1\nPrevious League Rank: 0\n\nPoints: 85\nThis Weeks Points: 85\n\nManager: Sonny Spencer\n\nStarting 11:\n\nPlayer Name: David Raya Martin, Player Points:8\nPlayer Name: Dan Burn, Player Points:5\nPlayer Name: William Saliba, Player Points:6\nPlayer Name: Trent Alexander-Arnold, Player Points:8\nPlayer Name: Anthony Gordon, Player Points:3\nPlayer Name: Amad Diallo, Player Points:3\nPlayer Name: Mohamed Salah, Player Points:14\nPlayer Name: Emile Smith Rowe, Player Points:3\nPlayer Name: Alexander Isak, Player Points:5\nPlayer Name: Erling Haaland, Player Points:7\nPlayer Name: Chris Wood, Player Points:9\n\nBench: \n\nPlayer Name: Łukasz Fabiański, Player Points:0\nPlayer Name: Valentín Barco, Player Points:0\nPlayer 

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
llm = ChatOpenAI()

In [16]:
class Report(BaseModel):
    headline:str = Field("The headline of the report")
    body:str = Field("The body of the report")

In [49]:
parser = PydanticOutputParser(pydantic_object=Report)
task_prompt = full_prompt_template.format(league_name=league_name, gameweek=gw_id, teams="\n".join(formatted_teams))
format_instrucions = parser.get_format_instructions()

prompt = PromptTemplate(
    template="{task_instructions}\n\n{format_instructions}",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser

generated_report = chain.invoke({"task_instructions": task_prompt})


In [52]:
client = Client()
collection = client.collection("reports")
doc = collection.document(str(gw_id)).get()
if doc.exists:
    doc_dict = doc.to_dict()
    if doc_dict.get("complete"):
        print(doc_dict)
else:
    collection.add(document_id=str(gw_id), document_data={
        "headline" : generated_report.headline,
        "body" : generated_report.body,
        "complete" : is_complete
    })

{'complete': True, 'headline': 'Major League Saka Leads European Sigma League in Week 1', 'body': "In the first week of the European Sigma League, Major League Saka, managed by Sonny Spencer, took an early lead with an impressive 85 points. The team's starting 11, including standout performances from Mohamed Salah (14 points) and Chris Wood (9 points), secured their position at the top of the league. Back of the Neto, managed by Julian Hoyle, closely followed with 73 points, thanks to strong showings from Bukayo Saka (12 points) and Erling Haaland (7 points). Øde-god, managed by Lucas Wood-Oliván, secured the third spot with 69 points, led by Bukayo Saka (12 points) and Trent Alexander-Arnold (8 points). Money Muniz, managed by Giles Usher, and 50 Shades of Gray, managed by Marc Tatam, rounded out the top five with 50 and 49 points respectively. Stay tuned for more updates as the European Sigma League progresses through its 38-week season."}


In [2]:
from app import entry_point
entry_point(None)

ModuleNotFoundError: No module named 'app'